# Fetch Papyrus data related to KLIFS and AlphaFold kinase structures

## Fetch and filter KLIFS structures

In [ ]:
# Select KLIFS filters (note that this also changes the number of AlphaFold structures as those are fetched without overlap)
KLIFS_MAX_RESOLUTION = 2.5
KLIFS_MAX_MISSING_RESIDUES = 5
KLIFS_MIN_QUALITY_SCORE = 8.0

In [ ]:
import base64
import gzip
import pandas as pd
import requests
from typing import Iterable, List

KLIFS_BASE_URL = 'https://klifs.net/api'
KLIFS_OUTPUT_CSV = 'klifs_filtered_structures_uniprot.csv'
KLIFS_OUTPUT_CSV_WITH_PROTEIN = 'klifs_filtered_structures_uniprot_with_protein_mol2.csv'
KLIFS_REQUEST_TIMEOUT = 60
KLIFS_KINASE_CHUNK_SIZE = 75

RUN_KLIFS_STAGE = True
SKIP_IF_KLIFS_OUTPUT_EXISTS = True

klifs_session = requests.Session()
klifs_session.headers.update({'Accept': 'application/json'})

def klifs_chunked(values: List[int], size: int) -> Iterable[List[int]]:
    for i in range(0, len(values), size):
        yield values[i:i + size]

def klifs_get_json(endpoint: str, params: dict | None = None):
    response = klifs_session.get(f"{KLIFS_BASE_URL}{endpoint}", params=params, timeout=KLIFS_REQUEST_TIMEOUT)
    response.raise_for_status()
    return response.json()

def fetch_human_kinase_information() -> pd.DataFrame:
    kinase_info = klifs_get_json('/kinase_information', params={'species': 'HUMAN'})
    kinase_df = pd.DataFrame(kinase_info)

    required = {'kinase_ID', 'uniprot'}
    missing = required - set(kinase_df.columns)
    if missing:
        raise ValueError(f"Missing required columns in kinase_information response: {missing}")

    keep_cols = [c for c in ['kinase_ID', 'name', 'species', 'uniprot'] if c in kinase_df.columns]
    kinase_df = kinase_df[keep_cols].copy()
    kinase_df['kinase_ID'] = pd.to_numeric(kinase_df['kinase_ID'], errors='coerce').astype('Int64')
    return kinase_df.dropna(subset=['kinase_ID'])

def fetch_structures_for_kinases(kinase_ids: List[int]) -> pd.DataFrame:
    all_structures = []
    for ids in klifs_chunked(kinase_ids, KLIFS_KINASE_CHUNK_SIZE):
        params = {'kinase_ID': ','.join(map(str, ids))}
        batch = klifs_get_json('/structures_list', params=params)
        if isinstance(batch, list):
            all_structures.extend(batch)
    return pd.DataFrame(all_structures)

def fetch_protein_mol2(structure_id: int) -> str:
    response = klifs_session.get(
        f"{KLIFS_BASE_URL}/structure_get_protein",
        params={'structure_ID': int(structure_id)},
        timeout=KLIFS_REQUEST_TIMEOUT,
        headers={'Accept': 'chemical/x-mol2, text/plain, */*'},
    )
    response.raise_for_status()
    return response.text

def compress_mol2_to_base64(mol2_text: str) -> str:
    compressed = gzip.compress(mol2_text.encode('utf-8'), compresslevel=9)
    return base64.b64encode(compressed).decode('ascii')

if RUN_KLIFS_STAGE:
    if SKIP_IF_KLIFS_OUTPUT_EXISTS and pd.io.common.file_exists(KLIFS_OUTPUT_CSV_WITH_PROTEIN):
        print(f"KLIFS stage skipped; existing output found: {KLIFS_OUTPUT_CSV_WITH_PROTEIN}")
    else:
        print('Running KLIFS acquisition stage...')
        print(
            f"Using filters: resolution <= {KLIFS_MAX_RESOLUTION}, "
            f"missing_residues <= {KLIFS_MAX_MISSING_RESIDUES}, "
            f"quality_score >= {KLIFS_MIN_QUALITY_SCORE}"
        )
        kinase_df = fetch_human_kinase_information()
        kinase_ids = kinase_df['kinase_ID'].dropna().astype(int).unique().tolist()
        structures_df = fetch_structures_for_kinases(kinase_ids)

        wanted_cols = [
            'structure_ID', 'kinase_ID', 'kinase', 'species', 'pdb', 'chain',
            'resolution', 'quality_score', 'missing_residues',
        ]
        present_cols = [c for c in wanted_cols if c in structures_df.columns]
        structures_df = structures_df[present_cols].copy()

        merged = structures_df.merge(
            kinase_df[['kinase_ID', 'uniprot']],
            on='kinase_ID',
            how='left',
        )

        for col in ['resolution', 'quality_score', 'missing_residues']:
            if col in merged.columns:
                merged[col] = pd.to_numeric(merged[col], errors='coerce')

        filtered = merged.loc[
            (merged['resolution'] <= KLIFS_MAX_RESOLUTION)
            & (merged['missing_residues'] <= KLIFS_MAX_MISSING_RESIDUES)
            & (merged['quality_score'] >= KLIFS_MIN_QUALITY_SCORE)
            & (merged['uniprot'].notna())
            & (merged['uniprot'].astype(str).str.strip() != '')
        ].copy()

        best_per_uniprot = (
            filtered
            .sort_values(
                by=['uniprot', 'quality_score', 'resolution', 'missing_residues'],
                ascending=[True, False, True, True],
                kind='mergesort',
            )
            .drop_duplicates(subset=['uniprot'], keep='first')
            .sort_values('uniprot')
            .reset_index(drop=True)
        )

        best_per_uniprot.to_csv(KLIFS_OUTPUT_CSV, index=False)
        print(f"✓ Saved {len(best_per_uniprot):,} KLIFS structures → {KLIFS_OUTPUT_CSV}")

        df_with_protein = best_per_uniprot.copy()
        df_with_protein['protein_mol2_gzip_base64'] = None
        df_with_protein['protein_mol2_fetch_error'] = None

        for i, structure_id in enumerate(df_with_protein['structure_ID'], start=1):
            try:
                mol2_text = fetch_protein_mol2(structure_id)
                df_with_protein.at[i - 1, 'protein_mol2_gzip_base64'] = compress_mol2_to_base64(mol2_text)
            except Exception as exc:
                df_with_protein.at[i - 1, 'protein_mol2_fetch_error'] = str(exc)

            if i % 25 == 0 or i == len(df_with_protein):
                print(f"KLIFS protein payload processed: {i}/{len(df_with_protein)}")

        df_with_protein.to_csv(KLIFS_OUTPUT_CSV_WITH_PROTEIN, index=False)
        print(f"✓ Saved {len(df_with_protein):,} KLIFS structures with protein payload → {KLIFS_OUTPUT_CSV_WITH_PROTEIN}")
else:
    print('KLIFS acquisition stage disabled (RUN_KLIFS_STAGE=False).')

KLIFS stage skipped; existing output found: klifs_filtered_structures_uniprot_with_protein_mol2.csv


## Configure Papyrus inputs and KLIFS accession list

In [ ]:
import pandas as pd
import requests
import os
import subprocess

DATA_DIR = 'data'
os.makedirs(DATA_DIR, exist_ok=True)

# File paths (raw data stored in data/)
PROTEIN_TARGETS_TSV = f'{DATA_DIR}/05.6_combined_set_protein_targets.tsv'
PROTEIN_TARGETS_XZ  = f'{DATA_DIR}/05.6_combined_set_protein_targets.tsv.xz'
BIOACTIVITIES_TSV   = f'{DATA_DIR}/05.6_combined_set_without_stereochemistry.tsv'
BIOACTIVITIES_XZ    = f'{DATA_DIR}/05.6_combined_set_without_stereochemistry.tsv.xz'
UNIPROT_TSV         = f'{DATA_DIR}/uniprot_human_kinases.tsv'
UNIPROT_GZ          = f'{DATA_DIR}/uniprot_human_kinases.tsv.gz'

PROTEIN_TARGETS_URL = 'https://zenodo.org/records/13817795/files/05.6_combined_set_protein_targets.tsv.xz'
BIOACTIVITIES_URL   = 'https://zenodo.org/records/13817795/files/05.6_combined_set_without_stereochemistry.tsv.xz'
UNIPROT_URL         = 'https://rest.uniprot.org/uniprotkb/stream?compressed=true&fields=accession%2Cid%2Cprotein_name%2Cgene_names%2Clength%2Cmass%2Cxref_chembl%2Cxref_hgnc%2Cgene_primary%2Cec&format=tsv&query=%28%28keyword%3AKW-0418%29+AND+%28reviewed%3Atrue%29%29+AND+%28model_organism%3A9606%29'

# KINASE_ACCESSIONS are derived from KLIFS output generated in the previous cell
if 'best_per_uniprot' in globals() and isinstance(best_per_uniprot, pd.DataFrame) and 'uniprot' in best_per_uniprot.columns:
    _klifs_uniprots = best_per_uniprot['uniprot']
elif os.path.exists(KLIFS_OUTPUT_CSV):
    _klifs_uniprots = pd.read_csv(KLIFS_OUTPUT_CSV, usecols=['uniprot'])['uniprot']
elif os.path.exists(KLIFS_OUTPUT_CSV_WITH_PROTEIN):
    _klifs_uniprots = pd.read_csv(KLIFS_OUTPUT_CSV_WITH_PROTEIN, usecols=['uniprot'])['uniprot']
else:
    raise FileNotFoundError(
        'KLIFS outputs not found. Run cell 2 first, or place KLIFS output CSV files in the notebook directory.'
    )

KINASE_ACCESSIONS = set(
    _klifs_uniprots.dropna().astype(str).str.strip().loc[lambda s: s != ''].tolist()
)

print(f"Data directory: {os.path.abspath(DATA_DIR)}")
print(f"KLIFS accessions loaded: {len(KINASE_ACCESSIONS)}")

Data directory: /home/wilkesbts/Documents/MEP/papyrus_klifs/data
KLIFS accessions loaded from stage 0: 225


## Utility functions for downloading and decompression

In [59]:
def download_file(url, local_path):
    print(f"Downloading {local_path}...")
    with requests.get(url, stream=True) as r:
        r.raise_for_status()
        with open(local_path, 'wb') as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)
    print("Done.")

def decompress(path, cmd='unxz'):
    print(f"Decompressing {path}...")
    subprocess.run([cmd, path], check=True)
    print("Done.")

## Load Papyrus and UniProt datasets

In [60]:
# Load protein targets
if not os.path.exists(PROTEIN_TARGETS_TSV):
    download_file(PROTEIN_TARGETS_URL, PROTEIN_TARGETS_XZ)
    decompress(PROTEIN_TARGETS_XZ)

df_protein_targets = pd.read_csv(PROTEIN_TARGETS_TSV, sep='\t', encoding='latin1')
print(f"Protein targets: {df_protein_targets.shape}")
print(df_protein_targets.head())

# Load UniProt human kinases
if not os.path.exists(UNIPROT_TSV):
    download_file(UNIPROT_URL, UNIPROT_GZ)
    decompress(UNIPROT_GZ, cmd='gunzip')

df_uniprot = pd.read_csv(UNIPROT_TSV, sep='\t', encoding='latin1')
print(f"\nUniProt kinases: {df_uniprot.shape}")
print(df_uniprot.head())

Protein targets: (7521, 9)
   target_id HGNC_symbol     UniProtID      Status  \
0  P47747_WT         NaN    HRH2_CAVPO    reviewed   
1  B0FL73_WT         NaN  B0FL73_CAVPO  unreviewed   
2  Q8K4Z4_WT         NaN   ADRB2_CAVPO    reviewed   
3  P97266_WT         NaN    OPRM_CAVPO    reviewed   
4  P41144_WT         NaN    OPRK_CAVPO    reviewed   

                       Organism  \
0  Cavia porcellus (Guinea pig)   
1  Cavia porcellus (Guinea pig)   
2  Cavia porcellus (Guinea pig)   
3  Cavia porcellus (Guinea pig)   
4  Cavia porcellus (Guinea pig)   

                                      Classification  Length  \
0  Membrane receptor->Family A G protein-coupled ...   359.0   
1  Membrane receptor->Family A G protein-coupled ...   467.0   
2  Membrane receptor->Family A G protein-coupled ...   418.0   
3  Membrane receptor->Family A G protein-coupled ...    98.0   
4  Membrane receptor->Family A G protein-coupled ...   380.0   

                                            Sequence

## Build KLIFS-matched Papyrus bioactivity subset

In [61]:
# Load bioactivities in chunks, filtering to KLIFS accessions and wildtype (WT) proteins
klifs_prefixes = {acc[:6] for acc in KINASE_ACCESSIONS}

if not os.path.exists(BIOACTIVITIES_TSV):
    download_file(BIOACTIVITIES_URL, BIOACTIVITIES_XZ)
    decompress(BIOACTIVITIES_XZ)

chunks = []
for i, chunk in enumerate(pd.read_csv(
    BIOACTIVITIES_TSV, sep='\t', encoding='latin1', chunksize=100_000,
    usecols=['Activity_ID', 'Quality', 'source', 'SMILES', 'target_id',
             'accession', 'Protein_Type', 'pchembl_value', 'Activity_class']
)):
    chunk = chunk[
        chunk['target_id'].str[:6].isin(klifs_prefixes) &
        (chunk['Protein_Type'] == 'WT')
    ]
    chunks.append(chunk)
    if (i + 1) % 20 == 0:
        print(f"  {(i + 1) * 100_000:,} rows processed...")

df_bioactivities = pd.concat(chunks, ignore_index=True)
print(f"Bioactivities loaded (KLIFS-filtered, WT only): {df_bioactivities.shape}")
print(df_bioactivities.head())

Done.
Decompressing data/05.6_combined_set_without_stereochemistry.tsv.xz...
Done.
  2,000,000 rows processed...


/tmp/ipykernel_215442/353075146.py:9: DtypeWarning: Columns (25) have mixed types. Specify dtype option on import or set low_memory=False.
  for i, chunk in enumerate(pd.read_csv(


  4,000,000 rows processed...
  6,000,000 rows processed...
  8,000,000 rows processed...
  10,000,000 rows processed...
  12,000,000 rows processed...
  14,000,000 rows processed...
  16,000,000 rows processed...
  18,000,000 rows processed...
  20,000,000 rows processed...
  22,000,000 rows processed...
  24,000,000 rows processed...
  26,000,000 rows processed...
  28,000,000 rows processed...
  30,000,000 rows processed...
  32,000,000 rows processed...
  34,000,000 rows processed...
  36,000,000 rows processed...
  38,000,000 rows processed...
  40,000,000 rows processed...
  42,000,000 rows processed...
  44,000,000 rows processed...
  46,000,000 rows processed...
  48,000,000 rows processed...
  50,000,000 rows processed...
  52,000,000 rows processed...
  54,000,000 rows processed...
  56,000,000 rows processed...
  58,000,000 rows processed...
Bioactivities loaded (KLIFS-filtered, WT only): (3032508, 9)
                   Activity_ID Quality                          source  \


## Filter kinase targets and interactions

In [62]:
# Filter protein_targets to UniProt kinases
uniprot_entry_names = set(df_uniprot['Entry Name'].unique())
df_human_kinase_targets = df_protein_targets[df_protein_targets['UniProtID'].isin(uniprot_entry_names)].copy()
print(f"Targets (UniProt filtered): {df_protein_targets.shape} → {df_human_kinase_targets.shape}")

# Filter KLIFS accessions
df_klifs_targets = df_human_kinase_targets[
    df_human_kinase_targets['target_id'].str[:6].isin(klifs_prefixes)
].copy()
print(f"Targets (KLIFS filtered):   {df_human_kinase_targets.shape} → {df_klifs_targets.shape}")

# Filter to wildtype (WT) only
df_klifs_targets = df_klifs_targets[df_klifs_targets['target_id'].str.endswith('_WT')].copy()
print(f"Targets (WT only):          → {df_klifs_targets.shape}")

# Filter bioactivities
klifs_target_ids = set(df_klifs_targets['target_id'].unique())
df_klifs_interactions = df_bioactivities[df_bioactivities['target_id'].isin(klifs_target_ids)].copy()
print(f"Interactions (KLIFS, WT):   {df_klifs_interactions.shape}")

Targets (UniProt filtered): (7521, 9) → (586, 9)
Targets (KLIFS filtered):   (586, 9) → (273, 9)
Targets (WT only):          → (217, 9)
Interactions (KLIFS, WT):   (3031927, 9)


## Report quality distribution and save KLIFS outputs

In [63]:
# Quality analysis: count total entries and non-missing values per quality tier
for qualities in [['Low', 'Medium', 'High'], ['Medium', 'High'], ['High']]:
    df_q = df_klifs_interactions[df_klifs_interactions['Quality'].isin(qualities)]
    print(f"[{'/'.join(qualities)}]  total: {len(df_q):,}")
    print(f"  pchembl_value  (non-null): {df_q['pchembl_value'].count():,}")
    print(f"  Activity_class (non-null): {df_q['Activity_class'].count():,}")
    print()

[Low/Medium/High]  total: 3,031,927
  pchembl_value  (non-null): 397,470
  Activity_class (non-null): 2,634,457

[Medium/High]  total: 279,237
  pchembl_value  (non-null): 279,237
  Activity_class (non-null): 0

[High]  total: 232,727
  pchembl_value  (non-null): 232,727
  Activity_class (non-null): 0



In [64]:
# Save high-quality bioactivities and KLIFS targets to CSV
df_high = df_klifs_interactions[df_klifs_interactions['Quality'] == 'High'].copy()
df_high.to_csv('papyrus_klifs_kinase_bioactivities_high_quality.csv', index=False)
print(f"✓ Saved {len(df_high):,} high-quality bioactivities → papyrus_klifs_kinase_bioactivities_high_quality.csv")

df_klifs_targets.to_csv('papyrus_klifs_kinase_targets.csv', index=False)
print(f"✓ Saved {len(df_klifs_targets):,} KLIFS targets → papyrus_klifs_kinase_targets.csv")

✓ Saved 232,727 high-quality bioactivities → papyrus_klifs_kinase_bioactivities_high_quality.csv
✓ Saved 217 KLIFS targets → papyrus_klifs_kinase_targets.csv


## Find non-KLIFS kinases and add AlphaFold structures

In [65]:
# High-quality Papyrus kinase data points whose UniProt accession is NOT in the KLIFS list
import base64
import gzip
import time

papyrus_kinase_target_ids = set(df_human_kinase_targets['target_id'].dropna().astype(str).unique())

non_klifs_chunks = []
for i, chunk in enumerate(pd.read_csv(
    BIOACTIVITIES_TSV,
    sep='\t',
    encoding='latin1',
    chunksize=100_000,
    usecols=[
        'Activity_ID', 'Quality', 'source', 'SMILES', 'target_id',
        'accession', 'Protein_Type', 'pchembl_value', 'Activity_class'
    ]
)):
    chunk = chunk[
        (chunk['Quality'] == 'High')
        & (chunk['Protein_Type'] == 'WT')
        & (chunk['target_id'].isin(papyrus_kinase_target_ids))
    ].copy()

    accession_from_col = chunk['accession'].where(chunk['accession'].notna())
    accession_from_col = accession_from_col.astype(str).str.strip()
    accession_from_col = accession_from_col.mask(accession_from_col.isin(['', 'nan', 'None']))
    accession_from_target = chunk['target_id'].astype(str).str.split('_').str[0]
    chunk['accession_final'] = accession_from_col.fillna(accession_from_target)

    chunk_non_klifs = chunk[~chunk['accession_final'].isin(klifs_prefixes)].copy()
    non_klifs_chunks.append(chunk_non_klifs)

    if (i + 1) % 20 == 0:
        print(f"  {(i + 1) * 100_000:,} rows processed...")

df_non_klifs_high = pd.concat(non_klifs_chunks, ignore_index=True)
unique_non_klifs_accessions = sorted(
    df_non_klifs_high['accession_final'].dropna().astype(str).str.strip().unique().tolist()
)

print(f"High-quality WT Papyrus kinase data points not in KLIFS: {len(df_non_klifs_high):,}")
print(f"Unique non-KLIFS kinase accessions: {len(unique_non_klifs_accessions):,}")

# Save non-KLIFS high-quality bioactivities
NON_KLIFS_BIOACTIVITIES_CSV = 'papyrus_non_klifs_kinase_bioactivities_high_quality.csv'
df_non_klifs_high.to_csv(NON_KLIFS_BIOACTIVITIES_CSV, index=False)
print(f"✓ Saved {len(df_non_klifs_high):,} rows → {NON_KLIFS_BIOACTIVITIES_CSV}")

# AlphaFold retrieval for non-KLIFS accessions
AF_BASE_URL = 'https://alphafold.ebi.ac.uk/api/prediction'
AF_STRUCTURES_WITH_PDB_CSV = 'alphafold_non_klifs_structures_uniprot_with_protein_pdb.csv'

def fetch_alphafold_prediction(uniprot_accession: str):
    url = f"{AF_BASE_URL}/{uniprot_accession}"
    response = requests.get(url, timeout=60)
    if response.status_code == 404:
        return []
    response.raise_for_status()
    payload = response.json()
    return payload if isinstance(payload, list) else []

def fetch_text(url: str) -> str:
    response = requests.get(url, timeout=60)
    response.raise_for_status()
    return response.text

def compress_text_to_base64(text: str) -> str:
    compressed = gzip.compress(text.encode('utf-8'), compresslevel=9)
    return base64.b64encode(compressed).decode('ascii')

def classify_plddt(mean_plddt):
    if pd.isna(mean_plddt):
        return None
    if mean_plddt >= 90:
        return 'Very high'
    if mean_plddt >= 70:
        return 'Confident'
    if mean_plddt >= 50:
        return 'Low'
    return 'Very low'

af_rows = []
for i, acc in enumerate(unique_non_klifs_accessions, start=1):
    try:
        records = fetch_alphafold_prediction(acc)
        if not records:
            af_rows.append({
                'uniprot': acc,
                'af_found': False,
                'af_fetch_error': None,
            })
        else:
            for rec in records:
                af_rows.append({
                    'uniprot': acc,
                    'af_found': True,
                    'entryId': rec.get('entryId'),
                    'uniprotAccession': rec.get('uniprotAccession'),
                    'uniprotId': rec.get('uniprotId'),
                    'modelEntityId': rec.get('modelEntityId'),
                    'latestVersion': rec.get('latestVersion'),
                    'modelCreatedDate': rec.get('modelCreatedDate'),
                    'globalMetricValue': rec.get('globalMetricValue'),
                    'fractionPlddtVeryLow': rec.get('fractionPlddtVeryLow'),
                    'fractionPlddtLow': rec.get('fractionPlddtLow'),
                    'fractionPlddtConfident': rec.get('fractionPlddtConfident'),
                    'fractionPlddtVeryHigh': rec.get('fractionPlddtVeryHigh'),
                    'pdbUrl': rec.get('pdbUrl'),
                    'cifUrl': rec.get('cifUrl'),
                    'af_fetch_error': None,
                })
    except Exception as exc:
        af_rows.append({
            'uniprot': acc,
            'af_found': False,
            'af_fetch_error': str(exc),
        })

    if i % 25 == 0 or i == len(unique_non_klifs_accessions):
        print(f"AlphaFold metadata processed: {i}/{len(unique_non_klifs_accessions)}")
    time.sleep(0.05)

df_af_structures = pd.DataFrame(af_rows)
if not df_af_structures.empty:
    for col in [
        'globalMetricValue',
        'fractionPlddtVeryLow',
        'fractionPlddtLow',
        'fractionPlddtConfident',
        'fractionPlddtVeryHigh',
        'latestVersion',
    ]:
        if col in df_af_structures.columns:
            df_af_structures[col] = pd.to_numeric(df_af_structures[col], errors='coerce')

    if 'modelCreatedDate' in df_af_structures.columns:
        df_af_structures['modelCreatedDate'] = pd.to_datetime(
            df_af_structures['modelCreatedDate'], errors='coerce'
        )

    if 'globalMetricValue' in df_af_structures.columns:
        df_af_structures['confidence_tier'] = df_af_structures['globalMetricValue'].apply(classify_plddt)

    # Keep one model per accession: highest globalMetricValue, then newest version/date
    df_af_structures = (
        df_af_structures
        .sort_values(
            ['uniprot', 'globalMetricValue', 'latestVersion', 'modelCreatedDate'],
            ascending=[True, False, False, False],
            kind='mergesort',
            na_position='last',
        )
        .drop_duplicates(subset=['uniprot'], keep='first')
        .sort_values(['uniprot'])
        .reset_index(drop=True)
    )

print(f"✓ Kept best AlphaFold model per accession: {len(df_af_structures):,} rows")

# Fetch PDB for each best model and save only one CSV with compressed structure payload
df_af_with_pdb = df_af_structures.copy()
df_af_with_pdb['protein_pdb_gzip_base64'] = None
df_af_with_pdb['protein_pdb_fetch_error'] = None

if 'pdbUrl' in df_af_with_pdb.columns:
    valid_idx = df_af_with_pdb.index[df_af_with_pdb['pdbUrl'].notna()].tolist()
    for j, idx in enumerate(valid_idx, start=1):
        pdb_url = str(df_af_with_pdb.at[idx, 'pdbUrl']).strip()
        if not pdb_url:
            continue
        try:
            pdb_text = fetch_text(pdb_url)
            df_af_with_pdb.at[idx, 'protein_pdb_gzip_base64'] = compress_text_to_base64(pdb_text)
        except Exception as exc:
            df_af_with_pdb.at[idx, 'protein_pdb_fetch_error'] = str(exc)

        if j % 25 == 0 or j == len(valid_idx):
            print(f"AlphaFold PDB processed: {j}/{len(valid_idx)}")
        time.sleep(0.05)

# Save csv
structure_payload_cols = [c for c in df_af_with_pdb.columns if c not in ['pdbUrl', 'cifUrl']]
df_af_with_pdb_saved = df_af_with_pdb[structure_payload_cols].copy()
df_af_with_pdb_saved.to_csv(AF_STRUCTURES_WITH_PDB_CSV, index=False)
print(f"✓ Saved {len(df_af_with_pdb_saved):,} AlphaFold rows with compressed structure payload → {AF_STRUCTURES_WITH_PDB_CSV}")

df_af_with_pdb_saved.head(10)

  2,000,000 rows processed...


/tmp/ipykernel_215442/1793365478.py:9: DtypeWarning: Columns (25) have mixed types. Specify dtype option on import or set low_memory=False.
  for i, chunk in enumerate(pd.read_csv(


  4,000,000 rows processed...
  6,000,000 rows processed...
  8,000,000 rows processed...
  10,000,000 rows processed...
  12,000,000 rows processed...
  14,000,000 rows processed...
  16,000,000 rows processed...
  18,000,000 rows processed...
  20,000,000 rows processed...
  22,000,000 rows processed...
  24,000,000 rows processed...
  26,000,000 rows processed...
  28,000,000 rows processed...
  30,000,000 rows processed...
  32,000,000 rows processed...
  34,000,000 rows processed...
  36,000,000 rows processed...
  38,000,000 rows processed...
  40,000,000 rows processed...
  42,000,000 rows processed...
  44,000,000 rows processed...
  46,000,000 rows processed...
  48,000,000 rows processed...
  50,000,000 rows processed...
  52,000,000 rows processed...
  54,000,000 rows processed...
  56,000,000 rows processed...
  58,000,000 rows processed...
High-quality WT Papyrus kinase data points not in KLIFS: 85,769
Unique non-KLIFS kinase accessions: 281
✓ Saved 85,769 rows → papyrus_n

,uniprot,af_found,entryId,uniprotAccession,uniprotId,modelEntityId,latestVersion,modelCreatedDate,globalMetricValue,fractionPlddtVeryLow,fractionPlddtLow,fractionPlddtConfident,fractionPlddtVeryHigh,af_fetch_error,confidence_tier,protein_pdb_gzip_base64,protein_pdb_fetch_error
0,A4QPH2,True,AF-A4QPH2-4-F1,A4QPH2-4,PI4P2_HUMAN,AF-A4QPH2-4-F1,6.0,2025-08-01 00:00:00+00:00,86.06,0.034,0.096,0.313,0.557,None,Confident,H4sIAAYzuWkC/8S923JduXIl+u6vWNFP3RF78eB+cT+xKG...,None
1,O00141,True,AF-O00141-3-F1,O00141-3,SGK1_HUMAN,AF-O00141-3-F1,6.0,2025-08-01 00:00:00+00:00,80.25,0.142,0.133,0.222,0.503,None,Confident,H4sIAAczuWkC/8R9225dOZLle3/FQT9NAykN75fuJ6Wsst...,None
2,O00142,True,AF-O00142-5-F1,O00142-5,KITM_HUMAN,AF-O00142-5-F1,6.0,2025-08-01 00:00:00+00:00,92.25,0.024,0.024,0.101,0.851,None,Very high,H4sIAAczuWkC/8W9224lR5It+N5fsXGezgFEjt8v3U8Uk5...,None
3,O00329,True,AF-O00329-F1,O00329,PK3CD_HUMAN,AF-O00329-F1,6.0,2025-08-01 00:00:00+00:00,87.94,0.036,0.083,0.192,0.689,None,Confident,H4sIAAczuWkC/8R923JdOZLdu7/ihJ/siDo07peZJxbFlj...,None
4,O00418,True,AF-O00418-F1,O00418,EF2K_HUMAN,AF-O00418-F1,6.0,2025-08-01 00:00:00+00:00,71.81,0.297,0.068,0.179,0.457,None,Confident,H4sIAAczuWkC/8S92Y5dOZIl+l5fcdBP3UC4X85D1ZOH66...,None
5,O00443,True,AF-O00443-F1,O00443,P3C2A_HUMAN,AF-O00443-F1,6.0,2025-08-01 00:00:00+00:00,71.88,0.265,0.064,0.332,0.339,None,Confident,H4sIAAgzuWkC/8R9226dObLefZ5iIVcJ0Evh+bD3lVrW2N...,None
6,O00750,True,AF-O00750-F1,O00750,P3C2B_HUMAN,AF-O00750-F1,6.0,2025-08-01 00:00:00+00:00,74.38,0.214,0.079,0.324,0.383,None,Confident,H4sIAAgzuWkC/8R9WW9lOY7m+/yKi3maAfJ6tC/dT07Hrb...,None
7,O00764,True,AF-O00764-F1,O00764,PDXK_HUMAN,AF-O00764-F1,6.0,2025-08-01 00:00:00+00:00,95.81,0.003,0.003,0.074,0.920,None,Very high,H4sIAAgzuWkC/8S923IdSY4l+j5fsW2eZsyKPH6/dD8xKZ...,None
8,O14578,True,AF-O14578-2-F1,O14578-2,CTRO_HUMAN,AF-O14578-2-F1,6.0,2025-08-01 00:00:00+00:00,85.25,0.085,0.075,0.230,0.610,None,Confident,H4sIAAkzuWkC/8S9225dOZImfN9PsTFXM0BKP8+H7iulrL...,None
9,O14730,True,AF-O14730-2-F1,O14730-2,RIOK3_HUMAN,AF-O14730-2-F1,6.0,2025-08-01 00:00:00+00:00,74.56,0.161,0.202,0.328,0.310,None,Confident,H4sIAAkzuWkC/8S9225dOZImfN9PsTFXM0Baw/Oh+0opq2...,None
